# SGLang integration test suite (Colab GPU)

Runs `tests/integrations_suite/test_sglang.py` on a GPU:

1. **Offline tests.** AuditKIT against a local server that answers exactly as SGLang's OpenAI server does (payloads follow sglang's `protocol.py`):
   - generation and token usage;
   - `reasoning_content` kept out of the answer;
   - `chat_template_kwargs`;
   - the OpenAI key never sent;
   - native tool calls, with string and dict arguments and without ids;
   - prompt mode;
   - per-sample errors and 503 retries;
   - the agent harness;
   - `compat.py` as a script.
2. **Live tests against real SGLang servers**, started one at a time:

   | Server | Model | Server flags | Live tests enabled |
   |---|---|---|---|
   | `qwen-tools` | Qwen2.5-1.5B-Instruct | `--tool-call-parser qwen25` | generation, prompt-mode tools, **native tools**, **harness** |
   | `qwen3-reasoning` | Qwen3-1.7B | `--tool-call-parser qwen25 --reasoning-parser qwen3` | the above, plus **reasoning kept out of the answer** and **`chat_template_kwargs` switching thinking off** |
   | `tiny-aya` | CohereLabs/tiny-aya-global (gated) | none (no tool slot in its template) | generation (multilingual), prompt-mode tools |
   | `tiny-aya-fire` | CohereLabs/tiny-aya-fire (gated) | none | the same, on a second `cohere2` checkpoint — the sibling models (Water, Earth) are the same architecture and need no separate run |
   | `command-r7b` | CohereLabs/c4ai-command-r7b-12-2024 (gated, 24 GB+ GPU) | `--tool-call-parser cohere_command4` | the above, plus **native tools through SGLang's Cohere parser**, checked to come back as structured `tool_calls` |

3. **`check_compat()` inside the SGLang environment.**
4. **One PASS / FAIL / SKIP / XFAIL table.** XFAIL is a known finding, recorded as a strict expected failure. If one turns into XPASS, it's been fixed and the marker should go.

**Runtime:** Colab **L4 or A100**. SGLang ≥ 0.5.11 needs an NVIDIA driver ≥ 580; on an older driver the install picks 0.5.10.

**Tokens:** the first code cell asks for them in **hidden input boxes**. What you paste is never written into the notebook file, so the `.ipynb` stays safe to share or commit.
- `GITHUB_TOKEN` is needed, to install AuditKIT.
- `HF_TOKEN` is needed only for the gated `tiny-aya` and `command-r7b` servers. Press Enter to skip it, and those servers are skipped.

**Time:** about 20–30 minutes, most of it the first SGLang install and three model downloads.

In [ ]:
# ---- settings ----
BRANCH = "main"   # once #40 is merged; to test #40 before it merges, use "fix/sglang-served-models"
PORT = 30000
SERVERS = {   # name -> (model, extra server args, env flags for the live tests, needs HF_TOKEN, min GPU GB)
    "qwen-tools": ("Qwen/Qwen2.5-1.5B-Instruct", ["--tool-call-parser", "qwen25"], {"SGLANG_TOOL_PARSER": "1"}, False, 0),
    "qwen3-reasoning": ("Qwen/Qwen3-1.7B", ["--tool-call-parser", "qwen25", "--reasoning-parser", "qwen3"],
                        {"SGLANG_TOOL_PARSER": "1", "SGLANG_REASONING": "1"}, False, 0),
    # Global and Fire are the same cohere2 architecture, so the second one is a
    # regression check that the registration is the architecture, not the id.
    "tiny-aya": ("CohereLabs/tiny-aya-global", [], {}, True, 0),
    "tiny-aya-fire": ("CohereLabs/tiny-aya-fire", [], {}, True, 0),
    # Cohere's action format (<|START_ACTION|>[...]<|END_ACTION|>) parsed into message.tool_calls by SGLang
    "command-r7b": ("CohereLabs/c4ai-command-r7b-12-2024", ["--tool-call-parser", "cohere_command4", "--context-length", "8192"],
                    {"SGLANG_TOOL_PARSER": "1"}, True, 23),
}
SERVER_TIMEOUT_S = 1200

In [ ]:
# ---- tokens: paste each one into the box that appears (hidden input; never saved in the notebook) ----
import os
from getpass import getpass
for key, why in (("GITHUB_TOKEN", "required: installs AuditKIT from the private repo"),
                 ("HF_TOKEN", "optional: the gated tiny-aya and command-r7b servers; Enter to skip")):
    if os.environ.get(key):
        print(f"{key}: already set in this runtime")
        continue
    value = getpass(f"Paste {key} ({why}): ").strip()
    if value:
        os.environ[key] = value
print({k: bool(os.environ.get(k)) for k in ("GITHUB_TOKEN", "HF_TOKEN")})

In [ ]:
# ---- install AuditKIT (an existing clone is moved to the branch head) ----
import os, subprocess, sys
REPO = "/content/AuditKIT-Internal"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    assert os.environ.get("GITHUB_TOKEN"), "paste GITHUB_TOKEN in the cell above"
    repo_url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, repo_url, REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "fetch", "-q", repo_url, BRANCH], check=True)
        subprocess.run(["git", "-C", REPO, "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", f"{REPO}[requests,dev]"], check=True)
else:
    REPO = os.path.abspath(os.path.join(os.getcwd(), *[".."] * 2)) if os.path.basename(os.getcwd()) == "integrations_suite" else os.getcwd()
print("repo:", REPO, "| commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
# Stop here, loudly, if this branch doesn't have the suite: otherwise every stage would report "no tests ran"
# and the results table would look like a clean pass.
if not os.path.exists(os.path.join(REPO, "tests/integrations_suite/test_sglang.py")):
    raise SystemExit(f"branch {BRANCH!r} has no SGLang suite (tests/integrations_suite/test_sglang.py). "
                     f'Set BRANCH = "fix/sglang-served-models" in the settings cell until #40 is merged, then re-run.')
# The Hugging Face token from any source (the HF_TOKEN secret, a huggingface_hub.login() or a cached token),
# exported so the gated-model check and the SGLang server process both see it.
try:
    from huggingface_hub import get_token
    if not os.environ.get("HF_TOKEN") and get_token():
        os.environ["HF_TOKEN"] = get_token()
except Exception:
    pass
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# ---- running the suite: one pytest call per stage, results collected from JUnit XML ----
import xml.etree.ElementTree as ET
import pandas as pd
RESULTS = []
TEST_FILE = "tests/integrations_suite/test_sglang.py"

def run_suite(stage, extra_env=None, select=None):
    xml = f"/tmp/sglang_{stage}.xml"
    env = {**os.environ, **(extra_env or {})}
    cmd = [sys.executable, "-m", "pytest", TEST_FILE, "-q", "-p", "no:cacheprovider", "-o", "addopts=", "-rxXs", "--color=no",
           f"--junitxml={xml}"] + (["-k", select] if select else [])
    out = subprocess.run(cmd, cwd=REPO, env=env, capture_output=True, text=True)
    print(out.stdout[-2500:] or out.stderr[-2500:])
    cases = list(ET.parse(xml).getroot().iter("testcase")) if os.path.exists(xml) else []
    # pytest exit codes: 0 all passed, 1 some failed; 2 interrupted, 3 internal error, 4 usage error
    # (e.g. the test file doesn't exist), 5 no tests collected. Anything but 0/1, or nothing
    # collected, is a FAIL row, never a silent empty stage.
    if out.returncode not in (0, 1) or not cases:
        RESULTS.append({"stage": stage, "test": "(pytest run)", "outcome": "FAIL",
                        "detail": f"pytest exit {out.returncode}: {'no tests collected' if not cases else 'error'}"})
    for case in cases:
        tags = {c.tag for c in case}
        skipped = next((c for c in case if c.tag == "skipped"), None)
        if "failure" in tags or "error" in tags:
            outcome = "FAIL"
        elif skipped is not None:
            outcome = "XFAIL" if (skipped.get("type") or "").endswith("xfail") else "SKIP"
        else:
            outcome = "PASS"
        RESULTS.append({"stage": stage, "test": case.get("name"), "outcome": outcome,
                        "detail": ((skipped.get("message") if skipped is not None else "") or "")[:120]})
    return out.returncode

## 1 · Offline tests

These need no GPU and no SGLang: AuditKIT against the SGLang-compatible local server. Two tests are **strict expected failures**, each recording a finding:
- an aborted request (`finish_reason="abort"`) is scored as an empty answer;
- a tool call that SGLang couldn't parse is read as "no call".

In [ ]:
run_suite("offline", select="not live")

## 2 · The SGLang environment

SGLang runs in its own Python 3.12 venv, built with uv; sglang's `outlines_core` pin has no cp313 wheel. The SGLang version is chosen from the NVIDIA driver, and `ninja` is installed because flashinfer compiles attention kernels on first use.

In [ ]:
SGL_ENV = "/content/sglang-env" if os.path.isdir("/content") else os.path.expanduser("~/sglang-env")
SGL_PY = f"{SGL_ENV}/bin/python"

def driver_major():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                             capture_output=True, text=True, check=True).stdout
        return int(out.split(".")[0])
    except (OSError, subprocess.CalledProcessError, ValueError):
        return None

drv = driver_major()
SGLANG_SPEC = "sglang>=0.5.18,<0.6" if drv is None or drv >= 580 else "sglang==0.5.10"
print("NVIDIA driver:", drv, "->", SGLANG_SPEC)

def env_python():
    if not os.path.exists(SGL_PY):
        return None
    r = subprocess.run([SGL_PY, "-c", "import sys; print('%d.%d' % sys.version_info[:2])"], capture_output=True, text=True)
    return r.stdout.strip() or None

if env_python() != "3.12":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.12", "--seed", "--clear", SGL_ENV], check=True)

def pip(*args):
    r = subprocess.run([SGL_PY, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"pip install {' '.join(args)} failed:\n" + (r.stderr or r.stdout)[-3000:])

pip("--upgrade", "pip"); pip(SGLANG_SPEC); pip("ninja")
print(subprocess.run([SGL_PY, "-c", "import sglang, torch; print('sglang', sglang.__version__, '| torch', torch.__version__)"],
                     capture_output=True, text=True).stdout)

In [ ]:
# check_compat() inside the SGLang venv (compat.py is stdlib-only, so it runs without AuditKIT installed there)
out = subprocess.run([SGL_PY, f"{REPO}/src/auditkit/compat.py", "--cuda"], capture_output=True, text=True)
print(out.stdout or out.stderr)
RESULTS.append({"stage": "sglang-env", "test": "check_compat (SGLang venv, --cuda)",
                "outcome": "PASS" if out.returncode == 0 else "FAIL", "detail": ""})

## 3 · Live tests against real SGLang servers

Each server starts from the SGLang venv with that venv's `bin/` on `PATH` (so flashinfer finds `ninja`), and falls back to the Triton attention backend if the kernel build fails. Then `test_sglang.py -k live` runs against it, and the server is stopped before the next one.

In [ ]:
import signal, time, requests
server_env = {**os.environ, "PATH": f"{SGL_ENV}/bin:" + os.environ.get("PATH", "")}
if "CUDA_HOME" not in server_env and os.path.isdir("/usr/local/cuda"):
    server_env["CUDA_HOME"] = "/usr/local/cuda"
NO_JIT_ARGS = ["--attention-backend", "triton", "--sampling-backend", "pytorch"]
JIT_ERRORS = ("ninja", "nvcc", "flashinfer/jit", "build_and_load", "CUDA_HOME")
LOG = "/tmp/sglang_server.log"

def log_tail(n=40):
    return "".join(open(LOG).readlines()[-n:])

def start(model, args):
    def launch(extra):
        cmd = [SGL_PY, "-m", "sglang.launch_server", "--model-path", model, "--host", "127.0.0.1", "--port", str(PORT), *extra]
        return subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True, env=server_env)
    def healthy(proc):
        deadline = time.time() + SERVER_TIMEOUT_S
        while time.time() < deadline:
            if proc.poll() is not None:
                return False
            try:
                if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5).ok:
                    return True
            except requests.RequestException:
                pass
            time.sleep(10)
        raise TimeoutError("server not healthy in time:\n" + log_tail())
    proc = launch(args)
    if not healthy(proc):
        if not any(k in log_tail(200) for k in JIT_ERRORS):
            raise RuntimeError("SGLang server exited:\n" + log_tail())
        print("flashinfer kernel build failed; retrying with the Triton backend")
        proc = launch([*args, *NO_JIT_ARGS])
        if not healthy(proc):
            raise RuntimeError("SGLang server exited (Triton backend too):\n" + log_tail())
    return proc

def stop(proc):
    os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    proc.wait(timeout=120)

import torch
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
for name, (model, args, flags, gated, min_gb) in SERVERS.items():
    print(f"\n==================== {name}: {model} {' '.join(args)}")
    if gated and not os.environ.get("HF_TOKEN"):
        print("skipped: needs HF_TOKEN (gated model)")
        RESULTS.append({"stage": name, "test": "(server)", "outcome": "SKIP", "detail": "needs HF_TOKEN"})
        continue
    if GPU_GB < min_gb:
        print(f"skipped: needs a {min_gb}+ GB GPU (this one: {GPU_GB:.0f} GB)")
        RESULTS.append({"stage": name, "test": "(server)", "outcome": "SKIP", "detail": f"needs {min_gb}+ GB GPU"})
        continue
    try:
        proc = start(model, args)
    except Exception as e:  # noqa: BLE001 -- record it, carry on with the next server
        print("server failed to start:", e)
        RESULTS.append({"stage": name, "test": "(server start)", "outcome": "FAIL", "detail": str(e)[-120:]})
        continue
    try:
        run_suite(name, {"SGLANG_BASE_URL": f"http://127.0.0.1:{PORT}/v1", "SGLANG_MODEL": model, **flags}, select="live")
    finally:
        stop(proc)

## 4 · Results

In [ ]:
df = pd.DataFrame(RESULTS)
display(df.pivot_table(index="stage", columns="outcome", values="test", aggfunc="count", fill_value=0))
display(df[df.outcome.isin(["FAIL", "XFAIL"])])
print("\nFAILED:", (df.outcome == "FAIL").sum(), "| known findings (XFAIL):", (df.outcome == "XFAIL").sum())

### Reading the results
- **FAIL:** a real problem. The pytest output above each stage has the traceback, and `/tmp/sglang_server.log` has the server side.
- **XFAIL:** a recorded finding: the two known offline ones. An XPASS would mean it's been fixed.
- **SKIP:** a test the server can't exercise; for example the reasoning tests on a server without `--reasoning-parser`, `tiny-aya` / `command-r7b` without `HF_TOKEN`, or `command-r7b` on a GPU under 24 GB.
- **`command-r7b` / `test_live_native_calls_come_back_structured`:** this is the live check that SGLang's `cohere_command4` parser works end to end, i.e. the `<|START_ACTION|>` markers survive decoding and come back as `message.tool_calls`. If it fails while `test_live_native_tool_calls` passes, the calls arrived only as raw text.